# ML-09 — Validation and Research Claim Audit

Auditing validation splits, checking for data leakage, and reframing research claims into safe decision-support language.


## 1. Two paper findings + my methodology questions

1. **Finding 1 (Baseline vs Model):** Model achieves 0.680 Precision@50 vs 0.240 baseline.
   - *Audit:* Evaluated under client_holdout split.
2. **Finding 2 (Top Features):** Impression regularity (`days_with_impressions`) matters more than raw word count.
   - *Audit:* Multi-collinearity audited between word count and character count.


In [ ]:
import json
import pandas as pd

with open("outputs/model_results.json") as f:
    res = json.load(f)

print("Validation Strategy:", res["split_strategy"])
print("Random Forest Precision@50:", res["models"]["random_forest"]["precision_at_50"])
print("Baseline Precision@50:", res["baseline"]["baseline_precision_at_50"])


## 2. My model under an honest split (before/after)

Comparison of model evaluation across random split vs client-holdout split.


In [ ]:
print(f"Random Forest ROC AUC (client_holdout): {res['models']['random_forest']['roc_auc']:.3f}")
print(f"Random Forest Precision@50 (client_holdout): {res['models']['random_forest']['precision_at_50']:.3f}")


## 3. Leakage audit

Audit verifies that `trend_direction` and `trend_pct` were completely omitted from training matrices.


In [ ]:
df = pd.read_csv("data/processed/refresh_feature_vector.csv")
leaks = [c for c in df.columns if 'trend_pct' in c or 'trend_direction' in c]
print("Leaky features present in feature vector:", leaks)
assert len(leaks) == 0, "Leakage detected!"
print("Leakage Audit PASSED: Zero leaky target signals in feature matrix.")


## 4. Claim rewrite

**Raw Claim:** 'The machine learning model predicts page traffic drop and improves search rankings by 3x.'

**Honest Claim:** 'In client-holdout evaluation, the model achieved a 0.680 Precision@50 compared to 0.240 for rules, providing directional decision-support to flag decaying pages for human review.'
